In [ ]:
```xml
<VSCode.Cell language="markdown">
# 📡 Event Streaming & CQRS Guide

**Building scalable systems with event sourcing, CQRS pattern, and event-driven architecture**

This guide covers:
- Event streaming concepts
- Event sourcing pattern
- CQRS (Command Query Responsibility Segregation)
- Event stores and event logs
- Stream processing
- Event-driven microservices
- Exactly-once semantics
- Event replay and projections
</VSCode.Cell>

<VSCode.Cell language="markdown">
## 1. Event Sourcing & Event Store

### Event Model
```python
from dataclasses import dataclass, field
from typing import Dict, List, Optional, Any
from datetime import datetime
from uuid import UUID
import json

@dataclass
class Event:
    """Core event structure"""
    event_id: str
    event_type: str  # e.g., "UserCreated", "OrderShipped"
    aggregate_id: str  # ID of the entity this event affects
    aggregate_type: str  # e.g., "User", "Order"
    timestamp: datetime
    version: int  # Event version in this aggregate
    payload: Dict[str, Any]
    metadata: Dict[str, Any] = field(default_factory=dict)
    
    def to_json(self) -> str:
        """Serialize event"""
        return json.dumps({
            'event_id': self.event_id,
            'event_type': self.event_type,
            'aggregate_id': self.aggregate_id,
            'aggregate_type': self.aggregate_type,
            'timestamp': self.timestamp.isoformat(),
            'version': self.version,
            'payload': self.payload,
            'metadata': self.metadata
        })

class EventStore:
    """Store and retrieve events"""
    
    def __init__(self):
        self.events: List[Event] = []
        self.snapshots: Dict[str, Dict] = {}  # aggregate_id -> snapshot
        self.subscriptions: List[callable] = []
    
    def append_event(self, event: Event) -> bool:
        """Append event to store"""
        
        # Check concurrency (optimistic locking)
        existing_version = self._get_latest_version(event.aggregate_id)
        
        if existing_version >= event.version:
            raise ConcurrencyError(f"Event version conflict: expected {event.version}, found {existing_version}")
        
        self.events.append(event)
        
        # Notify subscribers
        for subscriber in self.subscriptions:
            subscriber(event)
        
        return True
    
    def get_events(self, aggregate_id: str, from_version: int = 0) -> List[Event]:
        """Get events for aggregate"""
        return [e for e in self.events 
                if e.aggregate_id == aggregate_id and e.version > from_version]
    
    def get_event_stream(self) -> List[Event]:
        """Get all events in order"""
        return sorted(self.events, key=lambda e: e.timestamp)
    
    def _get_latest_version(self, aggregate_id: str) -> int:
        """Get latest version for aggregate"""
        events = self.get_events(aggregate_id)
        return max([e.version for e in events], default=0)
    
    def create_snapshot(self, aggregate_id: str, state: Dict, version: int):
        """Create snapshot for faster replay"""
        self.snapshots[aggregate_id] = {
            'state': state,
            'version': version,
            'created_at': datetime.now()
        }
    
    def restore_from_snapshot(self, aggregate_id: str) -> Optional[tuple]:
        """Restore aggregate state from snapshot"""
        if aggregate_id not in self.snapshots:
            return None
        
        snapshot = self.snapshots[aggregate_id]
        
        return (snapshot['state'], snapshot['version'])

class ConcurrencyError(Exception):
    """Concurrency conflict error"""
    pass

@dataclass
class AggregateRoot:
    """Base for event-sourced aggregates"""
    aggregate_id: str
    version: int = 0
    events: List[Event] = field(default_factory=list)
    
    def apply_event(self, event: Event):
        """Apply event to aggregate"""
        self.events.append(event)
        self.version = event.version
        
        # Call event handler
        handler_name = f"on_{event.event_type}"
        if hasattr(self, handler_name):
            getattr(self, handler_name)(event)
    
    def get_uncommitted_events(self) -> List[Event]:
        """Get events not yet persisted"""
        return self.events
    
    def mark_events_committed(self):
        """Clear uncommitted events"""
        self.events = []
```

### Event Replay
```python
class Projector:
    """Build read models from events"""
    
    def __init__(self):
        self.projections: Dict[str, Dict] = {}
    
    def project_from_events(self, events: List[Event], from_version: int = 0) -> Dict:
        """Build projection from event stream"""
        
        projection = {}
        
        for event in events:
            if event.version <= from_version:
                continue
            
            # Route to appropriate handler
            handler_name = f"project_{event.event_type}"
            
            if hasattr(self, handler_name):
                projection = getattr(self, handler_name)(projection, event)
        
        return projection
    
    def project_user_created(self, projection: Dict, event: Event) -> Dict:
        """Handle UserCreated event"""
        return {
            'user_id': event.aggregate_id,
            'email': event.payload.get('email'),
            'created_at': event.timestamp,
            'status': 'created'
        }
    
    def project_user_email_changed(self, projection: Dict, event: Event) -> Dict:
        """Handle UserEmailChanged event"""
        projection['email'] = event.payload.get('new_email')
        projection['email_updated_at'] = event.timestamp
        return projection
    
    def project_user_deactivated(self, projection: Dict, event: Event) -> Dict:
        """Handle UserDeactivated event"""
        projection['status'] = 'deactivated'
        projection['deactivated_at'] = event.timestamp
        return projection

class EventReplayer:
    """Replay events to reconstruct state"""
    
    def __init__(self, event_store: EventStore):
        self.event_store = event_store
    
    def replay_aggregate(self, aggregate_id: str, aggregate_class: type) -> object:
        """Replay aggregate from events"""
        
        aggregate = aggregate_class(aggregate_id)
        
        # Check for snapshot
        snapshot = self.event_store.restore_from_snapshot(aggregate_id)
        
        if snapshot:
            state, version = snapshot
            aggregate.state = state
            from_version = version
        else:
            from_version = 0
        
        # Replay events
        events = self.event_store.get_events(aggregate_id, from_version)
        
        for event in events:
            aggregate.apply_event(event)
        
        return aggregate
    
    def replay_all_aggregates(self, event_store: EventStore) -> Dict[str, object]:
        """Replay all aggregates"""
        aggregates = {}
        
        for event in event_store.get_event_stream():
            agg_id = event.aggregate_id
            
            if agg_id not in aggregates:
                aggregates[agg_id] = None  # Would instantiate correct type
        
        return aggregates
```
</VSCode.Cell>

<VSCode.Cell language="markdown">
## 2. CQRS Pattern

### Separation of Concerns
```python
class Command:
    """Base class for commands"""
    pass

class UserCreateCommand(Command):
    """Command to create user"""
    
    def __init__(self, user_id: str, email: str, name: str):
        self.user_id = user_id
        self.email = email
        self.name = name

class CommandHandler:
    """Process commands and generate events"""
    
    def __init__(self, event_store: EventStore):
        self.event_store = event_store
        self.handlers: Dict[type, callable] = {}
    
    def register_handler(self, command_type: type, handler: callable):
        """Register command handler"""
        self.handlers[command_type] = handler
    
    def execute(self, command: Command) -> bool:
        """Execute command"""
        handler = self.handlers.get(type(command))
        
        if not handler:
            raise ValueError(f"No handler for {type(command)}")
        
        return handler(command)

class Query:
    """Base class for queries"""
    pass

class GetUserQuery(Query):
    """Query to get user"""
    
    def __init__(self, user_id: str):
        self.user_id = user_id

class QueryHandler:
    """Answer queries from read models"""
    
    def __init__(self, read_models: Dict[str, Dict]):
        self.read_models = read_models
        self.handlers: Dict[type, callable] = {}
    
    def register_handler(self, query_type: type, handler: callable):
        """Register query handler"""
        self.handlers[query_type] = handler
    
    def execute(self, query: Query) -> Optional[Dict]:
        """Execute query"""
        handler = self.handlers.get(type(query))
        
        if not handler:
            raise ValueError(f"No handler for {type(query)}")
        
        return handler(query)

class CQRSBus:
    """Route commands and queries"""
    
    def __init__(self, event_store: EventStore):
        self.event_store = event_store
        self.command_handler = CommandHandler(event_store)
        self.query_handler = QueryHandler({})
    
    def send_command(self, command: Command) -> bool:
        """Send command"""
        return self.command_handler.execute(command)
    
    def send_query(self, query: Query) -> Optional[Dict]:
        """Send query"""
        return self.query_handler.execute(query)
```

### Read Model Synchronization
```python
class ReadModelSynchronizer:
    """Keep read models in sync with event stream"""
    
    def __init__(self, event_store: EventStore, projectors: List[Projector]):
        self.event_store = event_store
        self.projectors = projectors
        self.last_processed_position = 0
    
    def sync_read_models(self):
        """Process pending events and update read models"""
        
        events = self.event_store.get_event_stream()
        
        for event in events[self.last_processed_position:]:
            # Project to each read model
            for projector in self.projectors:
                projector.project_from_events([event])
            
            self.last_processed_position += 1
    
    def handle_event(self, event: Event):
        """Handle new event (real-time)"""
        
        for projector in self.projectors:
            projector.project_from_events([event])
```
</VSCode.Cell>

<VSCode.Cell language="markdown">
## 3. Stream Processing & Event Pipelines

### Event Processing
```python
class EventProcessor:
    """Process event streams"""
    
    def __init__(self):
        self.processors: List[callable] = []
        self.dead_letter_queue: List[Event] = []
    
    def add_processor(self, processor: callable):
        """Add event processor"""
        self.processors.append(processor)
    
    def process_event(self, event: Event) -> bool:
        """Process single event through pipeline"""
        
        try:
            for processor in self.processors:
                result = processor(event)
                
                if result is False:
                    self.dead_letter_queue.append(event)
                    return False
            
            return True
        except Exception as e:
            print(f"Error processing event: {e}")
            self.dead_letter_queue.append(event)
            return False
    
    def process_batch(self, events: List[Event]) -> Dict[str, int]:
        """Process batch of events"""
        
        stats = {
            'processed': 0,
            'failed': 0
        }
        
        for event in events:
            if self.process_event(event):
                stats['processed'] += 1
            else:
                stats['failed'] += 1
        
        return stats

class ExactlyOnceProcessor:
    """Ensure exactly-once processing"""
    
    def __init__(self):
        self.processed_events: set = set()  # event IDs
        self.idempotency_keys: Dict[str, Any] = {}
    
    def process_with_idempotency(self, event: Event, processor: callable) -> bool:
        """Process event with idempotency"""
        
        if event.event_id in self.processed_events:
            print(f"Event {event.event_id} already processed (duplicate)")
            return True  # Already processed
        
        # Check idempotency key
        idempotency_key = event.metadata.get('idempotency_key')
        
        if idempotency_key and idempotency_key in self.idempotency_keys:
            return True  # Already processed
        
        # Process event
        try:
            result = processor(event)
            
            self.processed_events.add(event.event_id)
            
            if idempotency_key:
                self.idempotency_keys[idempotency_key] = result
            
            return True
        except Exception as e:
            print(f"Processing failed: {e}")
            return False
```
</VSCode.Cell>

<VSCode.Cell language="markdown">
## 4. Event Streaming & CQRS Checklist

✅ **Event Design**
- [ ] Event types defined
- [ ] Event schema standardized
- [ ] Versioning strategy
- [ ] Payload structure
- [ ] Metadata captured

✅ **Event Store**
- [ ] Event persistence
- [ ] Immutability enforced
- [ ] Snapshots implemented
- [ ] Backup strategy
- [ ] Retention policy

✅ **CQRS Implementation**
- [ ] Commands defined
- [ ] Queries optimized
- [ ] Read models built
- [ ] Synchronization logic
- [ ] Consistency level

✅ **Stream Processing**
- [ ] Event processors
- [ ] Error handling
- [ ] Dead letter queue
- [ ] Monitoring alerts
- [ ] Scaling strategy

✅ **Operational**
- [ ] Event replay tested
- [ ] Disaster recovery
- [ ] Audit trail
- [ ] Performance metrics
- [ ] Troubleshooting guide
</VSCode.Cell>
```